Libraries

In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

Load Dataset. Bruker Panda for å loade datasettet i excel

In [22]:
def load_data(file_path):
    df = pd.read_csv(file_path)
    return df

In [23]:
file_path = '../../data/vehicles.csv'
df = pd.read_csv(file_path)
df = df.sample(n=10000, random_state=42)
df.head()

,id,url,region,region_url,price,year,manufacturer,model,condition,cylinders,...,size,type,paint_color,image_url,description,county,state,lat,long,posting_date
100905,7315883828,https://lakeland.craigslist.org/ctd/d/lakeland...,lakeland,https://lakeland.craigslist.org,36990,2017,ford,f150 super cab lariat,good,6 cylinders,...,NaN,pickup,white,https://images.craigslist.org/00s0s_lRS7etJoVE...,Carvana is the safer way to buy a car During t...,NaN,fl,28,-82,2021-05-02T15:31:06-0400
143835,7314599643,https://quadcities.craigslist.org/ctd/d/waterl...,"quad cities, IA/IL",https://quadcities.craigslist.org,27995,2006,chevrolet,corvette,good,8 cylinders,...,NaN,convertible,black,https://images.craigslist.org/00101_aa4DyXpKu0...,2006 *** Chevrolet Corvette Convertible Conver...,NaN,il,42,-92,2021-04-29T18:46:35-0500
20235,7308399808,https://littlerock.craigslist.org/ctd/d/clinto...,little rock,https://littlerock.craigslist.org,78423,2015,chevrolet,corvette,NaN,8 cylinders,...,NaN,convertible,NaN,https://images.craigslist.org/00A0A_kJsL7mVMCg...,➔ Want to see more pictures?Paste this link to...,NaN,ar,38,-94,2021-04-17T14:01:33-0500
300734,7312663807,https://wheeling.craigslist.org/ctd/d/follansb...,northern panhandle,https://wheeling.craigslist.org,14000,2013,bmw,328i,NaN,NaN,...,NaN,NaN,NaN,https://images.craigslist.org/00K0K_2oCjTKrjd9...,"**Deals, Deals, Deals** Beautiful 2013 BMW 3-S...",NaN,oh,40,-81,2021-04-25T23:53:42-0400
316249,7315368523,https://eugene.craigslist.org/ctd/d/cottage-gr...,eugene,https://eugene.craigslist.org,676,2019,chevrolet,suburban ls,NaN,8 cylinders,...,NaN,NaN,black,https://images.craigslist.org/00H0H_3hFsa4lTxO...,2019 Chevrolet Suburban LS Brads Chevy - ☎️ ...,NaN,or,44,-123,2021-05-01T10:04:24-0700


# Alle Features i datasettet

In [24]:
print('\n'.join(df.columns))

id
url
region
region_url
price
year
manufacturer
model
condition
cylinders
fuel
odometer
title_status
transmission
VIN
drive
size
type
paint_color
image_url
description
county
state
lat
long
posting_date


# Sjekke hvor mange verdier som er null

In [25]:
df.isnull().sum()

id                  0
url                 0
region              0
region_url          0
price               0
year               26
manufacturer      389
model             112
condition        4042
cylinders        4176
fuel               84
odometer          104
title_status      212
transmission       58
VIN              3795
drive            3010
size             7195
type             2116
paint_color      2990
image_url           0
description         0
county          10000
state               0
lat               157
long              157
posting_date        0
dtype: int64

Fjerne kolonner med mye støy slik som. County, Size, og kategorier med store hull kan utfylles med unknown: condition, cylinder, paint_color og drive. 

In [26]:
# Liste over kolonner med tekst/kategorier som har NaN
categorical_cols = ['drive', 'condition', 'cylinders', 'paint_color', 'fuel', 'transmission', 'type', 'title_status', 'manufacturer', 'model']

for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].fillna('unknown')

Fjerner også Image_url fordi xgboost får ikke benyttet å tolke bildet for maskinlæringen


In [27]:
df = df.drop(columns=['county', 'size', 'VIN', 'url', 'region_url', 'image_url', 'lat', 'long'], errors='ignore')

Split test data for later analysis

# Sjekker describe på odometer, price og year

In [28]:
import pandas as pd

# Viser flyttall som vanlige tall uten vitenskapelig notasjon
pd.set_option('display.float_format', lambda x: '%.0f' % x)

In [29]:
df[['odometer', 'price', 'year']].describe(
    percentiles=[0.01, 0.05, 0.1, 0.5, 0.9, 0.95, 0.99, 0.999]
)

,odometer,price,year
count,9896,10000,9974
mean,101767,17969,2011
std,279644,24947,9
min,0,0,1903
1%,5,0,1968
5%,6040,0,1999
10%,14455,700,2003
50%,86444,13988,2013
90%,177992,37902,2019
95%,202747,44995,2020


In [30]:
x = df.drop('price', axis=1)
y = df['price']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

df = x_train.copy()
df['price'] = y_train


Sjekker hvor mange biler som har  Odometer satt til 0


In [31]:
print("Nuller i odometer:", (df['odometer'] == 0).sum())

Nuller i odometer: 45


Fjerner radene hvor noen har 0 på kilometerstand

In [32]:
df = df.drop(df[df['odometer'] == 0].index)

In [33]:
if 'cylinders' in df.columns:
    df['cylinders'] = df['cylinders'].str.extract(r'(\d+)').astype(float)

In [34]:
print(df.dtypes)

id                int64
region              str
year            float64
manufacturer        str
model               str
condition           str
cylinders       float64
fuel                str
odometer        float64
title_status        str
transmission        str
drive               str
type                str
paint_color         str
description         str
state               str
posting_date        str
price             int64
dtype: object


In [35]:
df['year'] = df['year'].astype('Int64')

# Manglende verdier

In [36]:
for col in df.columns:
    number_null = df.loc[: , col].isnull().sum()
    percent_null = (number_null / len(df)) * 100
    print(' {} - {} - %{}'.format(col, number_null, round(percent_null, 3)))

 id - 0 - %0.0
 region - 0 - %0.0
 year - 21 - %0.264
 manufacturer - 0 - %0.0
 model - 0 - %0.0
 condition - 0 - %0.0
 cylinders - 3318 - %41.71
 fuel - 0 - %0.0
 odometer - 75 - %0.943
 title_status - 0 - %0.0
 transmission - 0 - %0.0
 drive - 0 - %0.0
 type - 0 - %0.0
 paint_color - 0 - %0.0
 description - 0 - %0.0
 state - 0 - %0.0
 posting_date - 0 - %0.0
 price - 0 - %0.0


Var lite manglende verdier


# Handtere duplikater

In [37]:
duplicate_rows = df.duplicated()

if duplicate_rows.any():
    print("There are duplicate rows in the DataFrame.")
else:
    print("There are no duplicate rows in the DataFrame.")

There are no duplicate rows in the DataFrame.


# Sjekke numeric kolonner

In [38]:
numeric_columns = df.select_dtypes(include=['int64', 'float64', 'Int64']).columns.tolist()

In [39]:
df[numeric_columns].describe()

,id,year,cylinders,odometer,price
count,7955,7934,4637,7880,7955
mean,7311460551,2011,6,104893,17910
std,4432155,9,2,310579,26674
min,7301594851,1927,3,1,0
25%,7308049352,2008,4,39979,5995
50%,7312694615,2013,6,88000,13900
75%,7315257276,2017,8,135584,25995
max,7317096685,2022,12,10000000,1234567


sjekke hva som er nummer hva

In [40]:
def num_checker(numIndex):
    values = np.sort(df[numeric_columns[numIndex]].unique())
    return numeric_columns[numIndex], values

In [41]:
num_checker(0)

('id',
 array([7301594851, 7301597966, 7301601024, ..., 7317089680, 7317091408,
        7317096685], shape=(7955,)))

In [42]:
num_checker(1)

('year',
 array([1927., 1929., 1931., 1932., 1934., 1936., 1939., 1940., 1947.,
        1948., 1949., 1950., 1951., 1952., 1953., 1954., 1955., 1956.,
        1957., 1958., 1959., 1960., 1961., 1962., 1963., 1964., 1965.,
        1966., 1967., 1968., 1969., 1970., 1971., 1972., 1973., 1974.,
        1975., 1976., 1977., 1978., 1979., 1980., 1981., 1982., 1983.,
        1984., 1985., 1986., 1987., 1988., 1989., 1990., 1991., 1992.,
        1993., 1994., 1995., 1996., 1997., 1998., 1999., 2000., 2001.,
        2002., 2003., 2004., 2005., 2006., 2007., 2008., 2009., 2010.,
        2011., 2012., 2013., 2014., 2015., 2016., 2017., 2018., 2019.,
        2020., 2021., 2022.,   nan]))

In [43]:
num_checker(2)

('cylinders', array([ 3.,  4.,  5.,  6.,  8., 10., 12., nan]))

In [44]:
num_checker(3)

('odometer',
 array([1.000000e+00, 2.000000e+00, 5.000000e+00, ..., 9.999999e+06,
        1.000000e+07,          nan], shape=(5873,)))

In [45]:
num_checker(4)

('price',
 array([      0,       1,       5, ..., 1000000, 1111111, 1234567],
       shape=(1935,)))

### Ser at det er noen outlier verdier vi kan fjerne slik som odonmeter på 400k og pris på over 150k

In [46]:
# Fjerne rader der pris er 0 eller ekstremt høy )
df = df[(df['price'] > 500) & (df['price'] < 150000)]

# Fjerne rader der odometer er 0 eller usannsynlig høyt
df = df[(df['odometer'] > 100) & (df['odometer'] < 400000)]

In [47]:
cat_columns = df.select_dtypes(exclude=['int64', 'float64', 'Int64']).columns.tolist()

In [48]:
def cat_checker(catIndex):
    values = np.sort(df[cat_columns[catIndex]].unique())
    return cat_columns[catIndex], values


In [49]:
cat_checker(0)

('region',
 array(['SF bay area', 'abilene', 'akron / canton', 'albany',
        'albuquerque', 'altoona-johnstown', 'amarillo', 'ames',
        'anchorage / mat-su', 'ann arbor', 'annapolis',
        'appleton-oshkosh-FDL', 'asheville', 'ashtabula', 'athens',
        'atlanta', 'auburn', 'augusta', 'austin', 'bakersfield',
        'baltimore', 'baton rouge', 'battle creek',
        'beaumont / port arthur', 'bellingham', 'bemidji', 'bend',
        'billings', 'binghamton', 'birmingham', 'bismarck', 'bloomington',
        'bloomington-normal', 'boise', 'boone', 'boston', 'boulder',
        'bowling green', 'bozeman', 'brainerd', 'brownsville', 'brunswick',
        'buffalo', 'butte', 'cape cod / islands', 'catskills',
        'cedar rapids', 'central NJ', 'central louisiana',
        'central michigan', 'champaign urbana', 'charleston', 'charlotte',
        'charlottesville', 'chattanooga', 'chautauqua', 'chicago', 'chico',
        'chillicothe', 'cincinnati', 'clarksville', 'cleveland

In [50]:
cat_checker(1)

('manufacturer',
 array(['acura', 'alfa-romeo', 'aston-martin', 'audi', 'bmw', 'buick',
        'cadillac', 'chevrolet', 'chrysler', 'dodge', 'ferrari', 'fiat',
        'ford', 'gmc', 'honda', 'hyundai', 'infiniti', 'jaguar', 'jeep',
        'kia', 'land rover', 'lexus', 'lincoln', 'mazda', 'mercedes-benz',
        'mercury', 'mini', 'mitsubishi', 'nissan', 'pontiac', 'porsche',
        'ram', 'rover', 'saturn', 'subaru', 'tesla', 'toyota', 'unknown',
        'volkswagen', 'volvo'], dtype=object))

In [51]:
cat_checker(2)

('model',
 array(["'99 H1 Hummer", '- forester', '1 series 128i', ...,
        'z4 sdrive35is roadster 2d', 'zdx', 'zl1 camaro'],
       shape=(2775,), dtype=object))

In [52]:
cat_checker(3)

('condition',
 array(['excellent', 'fair', 'good', 'like new', 'new', 'salvage',
        'unknown'], dtype=object))

In [53]:
cat_checker(4)

('fuel',
 array(['diesel', 'electric', 'gas', 'hybrid', 'other', 'unknown'],
       dtype=object))

In [54]:
cat_checker(5)

('title_status',
 array(['clean', 'lien', 'missing', 'parts only', 'rebuilt', 'salvage',
        'unknown'], dtype=object))

In [55]:
cat_checker(6)

('transmission',
 array(['automatic', 'manual', 'other', 'unknown'], dtype=object))

In [56]:
cat_checker(7)

('drive', array(['4wd', 'fwd', 'rwd', 'unknown'], dtype=object))

In [57]:
cat_checker(8)

('type',
 array(['SUV', 'bus', 'convertible', 'coupe', 'hatchback', 'mini-van',
        'offroad', 'other', 'pickup', 'sedan', 'truck', 'unknown', 'van',
        'wagon'], dtype=object))

In [58]:
cat_checker(9)

('paint_color',
 array(['black', 'blue', 'brown', 'custom', 'green', 'grey', 'orange',
        'purple', 'red', 'silver', 'unknown', 'white', 'yellow'],
       dtype=object))

# Checkpoint 

In [59]:
%store x_test
%store y_test
%store df
%store numeric_columns
%store cat_columns


Stored 'x_test' (DataFrame)
Stored 'y_test' (Series)
Stored 'df' (DataFrame)
Stored 'numeric_columns' (list)
Stored 'cat_columns' (list)
